### 🎯 Module Overview
This module covers everything you need to know about parsing and ingesting data for RAG systems, from basic text files to complex PDFs and databases. We'll use LangChain v1.2 and explore each technique with practical examples.

Table of Contents

- Introduction to Data Ingestion
- Text Files (.txt)
- PDF Documents
- Microsoft Word Documents
- CSV and Excel Files
- JSON and Structured Data
- Web Scraping
- Databases (SQL)
- Audio and Video Transcripts
- Advanced Techniques
- Best Practices

## Intoduction to Data Ingestion

In [1]:
from typing import List, Dict, Any
import pandas as pd
from langchain_core.documents import Document

In [5]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter, 
    CharacterTextSplitter,
    TokenTextSplitter)


### Understanding Document Structure In Langchain

In [7]:
## create a simple ducument
doc = Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "Ravi Shankar Kushwaha",
        "date_created": "2026-07-03",
        "cutom_field": "any_value"
    }
)
print("Document Structure")

print(f"Content : {doc.page_content}")
print(f"Metadata : {doc.metadata}")

# Why metadata matters:
print("\n Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")


Document Structure
Content : This is the main text content that will be embedded and searched.
Metadata : {'source': 'example.txt', 'page': 1, 'author': 'Ravi Shankar Kushwaha', 'date_created': '2026-07-03', 'cutom_field': 'any_value'}

 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


In [8]:
type(doc)

langchain_core.documents.base.Document

### Text files (.txt) - The Simplest Case {#2-text-files}

In [10]:
## Create a simple txt file
import os
os.makedirs("data/text_files", exist_ok=True)

In [11]:
sample_texts = {
    "data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simple, 
English-like syntax and high readability. Created by Guido van Rossum in 1991, 
it is now one of the world's most popular languages, widely used for web development, 
data science, artificial intelligence, and automation. 

Key Features
- Simple Syntax: Uses indentation to define code blocks instead of curly brackets, making the code cleaner and easier to read.
- Interpreted Language: Code is executed line-by-line, which simplifies debugging and allows for rapid prototyping.
- Dynamically Typed: You do not need to declare variable types (like int or string) before using them; Python identifies the type at runtime.
- Extensive Libraries: Python has a massive "standard library" and thousands of third-party packages (via PyPI) for tasks like data analysis (Pandas) and web development (Django).

Fundamental Concepts
- Variables and Data Types: Store information using basic types like int (whole numbers), float (decimals), str (text), and bool (True/False).
- Operators: Perform calculations and comparisons using symbols like +, -, *, /, and comparison operators like == (equal to) or != (not equal to).
- Control Flow: Use if, elif, and else statements for decision-making, and for or while loops to repeat actions.
- Data Structures: Organize groups of data using Lists (ordered, changeable), Tuples (ordered, unchangeable), Dictionaries (key-value pairs), and Sets (unordered unique items).
- Functions: Use the def keyword to create reusable blocks of code that perform specific tasks. 
""",
"data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that focuses on the development of computer programs that can access data and use it to learn for themselves. The process of learning begins with observations or data, such as from images, text, or sensor data, and involves identifying patterns in the data to make predictions or decisions without being explicitly programmed to do so.

Key Concepts
- Training Data: The dataset used to train the machine learning model.
- Features: The input variables used by the model to make predictions.
- Labels: The output variable that the model aims to predict.
- Model: A mathematical representation of the relationship between features and labels.
- Evaluation: The process of assessing the performance of the trained model.

Types of Machine Learning
- Supervised Learning: Learning from labeled training data.
- Unsupervised Learning: Finding hidden patterns in unlabeled data.
- Reinforcement Learning: Learning through interaction with an environment to maximize a reward signal.

Applications
Machine learning is used in various fields such as healthcare (diagnosis), finance (fraud detection), marketing (customer segmentation), and autonomous vehicles (object detection).
"""
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print(" Sample text files created in 'data/text_files' directory.")

 Sample text files created in 'data/text_files' directory.


### TextLoader - Read Single File

In [15]:
from langchain_community.document_loaders import TextLoader

# Loading single text file from the text files
loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load()
print(f"Loaded {len(documents)} document(s) from the text file.")
print(f"Document content:\n{documents[0].page_content[:100]}...")
print(f"Document metadata:\n{documents[0].metadata}")

Loaded 1 document(s) from the text file.
Document content:
Python Programming Introduction

Python is a high-level, interpreted programming language known for ...
Document metadata:
{'source': 'data/text_files/python_intro.txt'}


### Directory Loader- Multiple Text Files

In [17]:
from langchain_community.document_loaders import DirectoryLoader

## load all text files from the directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern to match files
    loader_cls=TextLoader, ##loader class to use
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents=dir_loader.load()

print(f"Loaded {len(documents)} document(s) from the directory.")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f" Source: {doc.metadata['source']}")
    print(f" Length: {len(doc.page_content)} characters")


# 📊 Analysis
print("\n📊 DirectoryLoader Characteristics:")
print("✅ Advantages:")
print("  - Loads multiple files at once")
print("  - Supports glob patterns")
print("  - Progress tracking")
print("  - Recursive directory scanning")

print("\n❌ Disadvantages:")
print("  - All files must be same type")
print("  - Limited error handling per file")
print("  - Can be memory intensive for large directories")

100%|██████████| 2/2 [00:00<00:00, 153.34it/s]

Loaded 2 document(s) from the directory.

Document 1:
 Source: data/text_files/python_intro.txt
 Length: 1614 characters

Document 2:
 Source: data/text_files/machine_learning.txt
 Length: 1254 characters

📊 DirectoryLoader Characteristics:
✅ Advantages:
  - Loads multiple files at once
  - Supports glob patterns
  - Progress tracking
  - Recursive directory scanning

❌ Disadvantages:
  - All files must be same type
  - Limited error handling per file
  - Can be memory intensive for large directories


### Text Spliting Statergies

In [19]:
### Different text splitting strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simple, \nEnglish-like syntax and high readability. Created by Guido van Rossum in 1991, \nit is now one of the world\'s most popular languages, widely used for web development, \ndata science, artificial intelligence, and automation. \n\nKey Features\n- Simple Syntax: Uses indentation to define code blocks instead of curly brackets, making the code cleaner and easier to read.\n- Interpreted Language: Code is executed line-by-line, which simplifies debugging and allows for rapid prototyping.\n- Dynamically Typed: You do not need to declare variable types (like int or string) before using them; Python identifies the type at runtime.\n- Extensive Libraries: Python has a massive "standard library" and thousands of third-party packages (via PyPI) for tasks like data analysis (Pandas) and web development (D

#### Method 1 - Character-based splitting

In [20]:
text=documents[0].page_content
text

'Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simple, \nEnglish-like syntax and high readability. Created by Guido van Rossum in 1991, \nit is now one of the world\'s most popular languages, widely used for web development, \ndata science, artificial intelligence, and automation. \n\nKey Features\n- Simple Syntax: Uses indentation to define code blocks instead of curly brackets, making the code cleaner and easier to read.\n- Interpreted Language: Code is executed line-by-line, which simplifies debugging and allows for rapid prototyping.\n- Dynamically Typed: You do not need to declare variable types (like int or string) before using them; Python identifies the type at runtime.\n- Extensive Libraries: Python has a massive "standard library" and thousands of third-party packages (via PyPI) for tasks like data analysis (Pandas) and web development (Django).\n\nFundamental Concepts\n- Variables and Data Types: Store information u

In [28]:
# Method 1 - Character-based splitting
print(" CHARACTER TEXT SPLITTER ")
char_splitter = CharacterTextSplitter(
    separator=" ", # Split on double newlines
    chunk_size=200, # Max characters per chunk
    chunk_overlap=20, # Overlap between chunks
    length_function=len # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

 CHARACTER TEXT SPLITTER 
Created 9 chunks
First chunk: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...


In [29]:
print(char_chunks[0])
print("------------")
print(char_chunks[1])
print("------------")
print(char_chunks[2])
print("------------")
print(char_chunks[3])

Python Programming Introduction

Python is a high-level, interpreted programming language known for its simple, 
English-like syntax and high readability. Created by Guido van Rossum in 1991, 
it is
------------
in 1991, 
it is now one of the world's most popular languages, widely used for web development, 
data science, artificial intelligence, and automation. 

Key Features
- Simple Syntax: Uses indentation
------------
Uses indentation to define code blocks instead of curly brackets, making the code cleaner and easier to read.
- Interpreted Language: Code is executed line-by-line, which simplifies debugging and
------------
debugging and allows for rapid prototyping.
- Dynamically Typed: You do not need to declare variable types (like int or string) before using them; Python identifies the type at runtime.
- Extensive


In [30]:
# Method 1 - Character-based splitting
print(" CHARACTER TEXT SPLITTER ")
char_splitter = CharacterTextSplitter(
    separator="\n", # Split on double newlines
    chunk_size=200, # Max characters per chunk
    chunk_overlap=20, # Overlap between chunks
    length_function=len # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

 CHARACTER TEXT SPLITTER 
Created 11 chunks
First chunk: Python Programming Introduction
Python is a high-level, interpreted programming language known for i...


In [31]:
print(char_chunks[0])
print("------------")
print(char_chunks[1])
print("------------")
print(char_chunks[2])
print("------------")
print(char_chunks[3])

Python Programming Introduction
Python is a high-level, interpreted programming language known for its simple, 
English-like syntax and high readability. Created by Guido van Rossum in 1991,
------------
it is now one of the world's most popular languages, widely used for web development, 
data science, artificial intelligence, and automation. 
Key Features
------------
Key Features
- Simple Syntax: Uses indentation to define code blocks instead of curly brackets, making the code cleaner and easier to read.
------------
- Interpreted Language: Code is executed line-by-line, which simplifies debugging and allows for rapid prototyping.


#### Method 2 - Recursive Character splitting (RECOMMENDED)

In [32]:
#### Method 2 - Recursive Character splitting (RECOMMENDED)
print(" RECURSIVE CHARACTER TEXT SPLITTER ")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "], # Try these separators in order
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")

 RECURSIVE CHARACTER TEXT SPLITTER 
Created 9 chunks
First chunk: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...


In [33]:
print(recursive_chunks[0])
print("------------")
print(recursive_chunks[1])
print("------------")
print(recursive_chunks[2])
print("------------")
print(recursive_chunks[3])

Python Programming Introduction

Python is a high-level, interpreted programming language known for its simple, 
English-like syntax and high readability. Created by Guido van Rossum in 1991, 
it is
------------
in 1991, 
it is now one of the world's most popular languages, widely used for web development, 
data science, artificial intelligence, and automation. 

Key Features
- Simple Syntax: Uses
------------
Simple Syntax: Uses indentation to define code blocks instead of curly brackets, making the code cleaner and easier to read.
- Interpreted Language: Code is executed line-by-line, which simplifies
------------
which simplifies debugging and allows for rapid prototyping.
- Dynamically Typed: You do not need to declare variable types (like int or string) before using them; Python identifies the type at


In [34]:
#### Method 2 - Recursive Character splitting (RECOMMENDED)
print(" RECURSIVE CHARACTER TEXT SPLITTER ")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], # Try these separators in order
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")

 RECURSIVE CHARACTER TEXT SPLITTER 
Created 12 chunks
First chunk: Python Programming Introduction...


In [35]:
print(recursive_chunks[0])
print("------------")
print(recursive_chunks[1])
print("------------")
print(recursive_chunks[2])
print("------------")
print(recursive_chunks[3])

Python Programming Introduction
------------
Python is a high-level, interpreted programming language known for its simple, 
English-like syntax and high readability. Created by Guido van Rossum in 1991,
------------
it is now one of the world's most popular languages, widely used for web development, 
data science, artificial intelligence, and automation.
------------
Key Features
- Simple Syntax: Uses indentation to define code blocks instead of curly brackets, making the code cleaner and easier to read.


In [36]:
# Create text without netural break points
simple_text = "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three and it is also quite long. This is sentence four and it is also quite long."

splitter = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}: '{chunks[i+1]}'")

    print()


Simple text example - 3 chunks:

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also'
Chunk 2: 'two and it is also quite long. This is sentence three and it is also quite'

Chunk 2: 'two and it is also quite long. This is sentence three and it is also quite'
Chunk 3: 'it is also quite long. This is sentence four and it is also quite long.'



### Method 3 - Token-based Splitting

In [38]:
# Method 3 - Token-based splitting
print(" TOKEN TEXT SPLITTER ")
token_splitter = TokenTextSplitter(
    chunk_size=50, # Size in tokens, not characters
    chunk_overlap=10, # Overlap in tokens
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")

 TOKEN TEXT SPLITTER 
Created 10 chunks
First chunk: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...


In [39]:
# 📊 Comparison
print("\n📊 Text Splitting Methods Comparison:")
print("\nCharacterTextSplitter:")
print("  ✅ Simple and predictable")
print("  ✅ Good for structured text")
print("  ❌ May break mid-sentence")
print("  Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("  ✅ Respects text structure")
print("  ✅ Tries multiple separators")
print("  ✅ Best general-purpose splitter")
print("  ❌ Slightly more complex")
print("  Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("  ✅ Respects model token limits")
print("  ✅ More accurate for embeddings")
print("  ❌ Slower than character-based")
print("  Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
  ✅ Simple and predictable
  ✅ Good for structured text
  ❌ May break mid-sentence
  Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
  ✅ Respects text structure
  ✅ Tries multiple separators
  ✅ Best general-purpose splitter
  ❌ Slightly more complex
  Use when: Default choice for most texts

TokenTextSplitter:
  ✅ Respects model token limits
  ✅ More accurate for embeddings
  ❌ Slower than character-based
  Use when: Working with token-limited models
